#### **Draft - Simple MLP and masks**
##### 20 April 2026

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy
import pickle
import pandas as pd
import numpy as np

### Implementation of a simple MLP

In [2]:
class MLP(nn.Module):
    def __init__(self, input_size=14*14, interm_size_1=64, interm_size_2=128, interm_size_3=256, num_classes=10):
        super(MLP, self).__init__()

        self.fc1 = nn.Linear(input_size, interm_size_1)
        self.fc2 = nn.Linear(interm_size_1, interm_size_2)
        self.fc3 = nn.Linear(interm_size_2, interm_size_3)
        self.fc4 = nn.Linear(interm_size_3, num_classes)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten input
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x

In [3]:
model= MLP()
for name, module in model.named_modules():
    print(name, module)

 MLP(
  (fc1): Linear(in_features=196, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=256, bias=True)
  (fc4): Linear(in_features=256, out_features=10, bias=True)
)
fc1 Linear(in_features=196, out_features=64, bias=True)
fc2 Linear(in_features=64, out_features=128, bias=True)
fc3 Linear(in_features=128, out_features=256, bias=True)
fc4 Linear(in_features=256, out_features=10, bias=True)


### Implementation of a "MaskLayer"

In [4]:
class MaskLayer(nn.Module):
    def __init__(self, mask_params, probs=False):
        super(MaskLayer, self).__init__()

        self.probs = probs

        if self.probs and (torch.any(mask_params > 1) or torch.any(mask_params < 0)):
            assert False

        self.mask_params = nn.Parameter(mask_params, requires_grad=True)

    def forward(self, x):

        if self.probs:
            # boolean_mask_params = ((self.mask_params >= 0.5).float() - self.mask_params).detach() + self.mask_params    # "detach trick" for straight-through estimator
            boolean_mask_params = (self.mask_params >= 0.5).float()
        else:
            boolean_mask_params = self.mask_params

        masked_output = x * boolean_mask_params     # The output of each neuron in the masked layer (after activations)
                                                    # is multiplied by the corresponding (boolean) element of the mask.

        return masked_output

**Note:** we implement the masks in such a way to reproduce the behavior of "*structured pruning*". It means that each element in a mask corresponds to a neuron in the original layer that is being masked.

### Implementation of a masked version of the MLP model

In [5]:
class MaskedMLP(nn.Module):
    def __init__(self, model, dict_mask_params, probs=False):
        super(MaskedMLP, self).__init__()

        # Note: 'probs' = True means that the mask params in dict_mask_params are between 0 and 1.
        #       'probs' = False means that the mask params are boolean (0 or 1).

        self.fc1 = copy.deepcopy(model.fc1)
        self.mask1 = MaskLayer(dict_mask_params['fc1'], probs)

        self.fc2 = copy.deepcopy(model.fc2)
        self.mask2 = MaskLayer(dict_mask_params['fc2'], probs)

        self.fc3 = copy.deepcopy(model.fc3)
        self.mask3 = MaskLayer(dict_mask_params['fc3'], probs)

        self.fc4 = copy.deepcopy(model.fc4) # We don't apply a mask to fc4, as we don't want to prune entire neurons in the last layer 

        # Saving mask layers in a list
        self.mask_layers = [self.mask1, self.mask2, self.mask3]


    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten input

        x = F.relu(self.fc1(x))
        x = self.mask1(x)           # In our implementation, we are applying the mask after the activations

        x = F.relu(self.fc2(x))
        x = self.mask2(x)

        x = F.relu(self.fc3(x))
        x = self.mask3(x)

        x = self.fc4(x)

        return x

#### Instantiating an MLP model (and saving it, without masks)

In [6]:
model = MLP()
torch.save(model.state_dict(), "mlp_model.pth")

model

MLP(
  (fc1): Linear(in_features=196, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=256, bias=True)
  (fc4): Linear(in_features=256, out_features=10, bias=True)
)

#### Creating the masks

In [7]:
def generate_random_vector(N, p):
    # (You can ignore the implementation details here. This function is just used to generate random masks, in our example.)
    # This function generates a vector of length N, with random values between 0 and 1,
    # and such that a percentage p (0 <= p <= 1) of its elements are smaller than 0.5.

    m = torch.rand(N) < p

    x = torch.zeros(N)
    x[m] = torch.rand(m.sum()) * 0.5
    x[~m] = 0.5 + torch.rand((~m).sum()) * 0.5

    return x



# Let's suppose that we want:
#   percentage of pruned neurons in layer fc1: 20%
#   percentage of pruned neurons in layer fc2: 40%
#   percentage of pruned neurons in layer fc3: 30%

dict_mask_params_probs = {
    'fc1': generate_random_vector(N=model.fc1.out_features, p=0).requires_grad_(True),
    'fc2': generate_random_vector(N=model.fc2.out_features, p=0.8).requires_grad_(True),
    'fc3': generate_random_vector(N=model.fc3.out_features, p=0.6).requires_grad_(True),
    'fc4': generate_random_vector(N=model.fc4.out_features, p=0).requires_grad_(True)
}

# Boolean ("binarized") version of the masks
dict_mask_params_boolean = {k:(v >= 0.5).float() for k,v in dict_mask_params_probs.items()}


# Saving the "probability" and "boolean" versions of the masks
with open('dict_mask_params_probs.pkl', 'wb') as f:
    pickle.dump(dict_mask_params_probs, f)
with open('dict_mask_params_boolean.pkl', 'wb') as f:
    pickle.dump(dict_mask_params_boolean, f)

In [8]:
dict_mask_params_probs

{'fc1': tensor([0.8919, 0.9331, 0.6246, 0.9536, 0.9385, 0.5719, 0.5877, 0.5235, 0.7362,
         0.6367, 0.5371, 0.8091, 0.6430, 0.6810, 0.7832, 0.5742, 0.5045, 0.6753,
         0.9494, 0.6054, 0.8593, 0.6320, 0.7636, 0.8954, 0.6625, 0.9302, 0.9570,
         0.6790, 0.7952, 0.6679, 0.6066, 0.6719, 0.8722, 0.5808, 0.9054, 0.8321,
         0.6886, 0.5780, 0.6424, 0.5479, 0.6319, 0.6395, 0.5034, 0.8391, 0.5948,
         0.9612, 0.5109, 0.9718, 0.8679, 0.6950, 0.8529, 0.6165, 0.7791, 0.9355,
         0.7241, 0.8247, 0.5309, 0.8344, 0.8830, 0.6450, 0.7333, 0.7359, 0.6545,
         0.5027], requires_grad=True),
 'fc2': tensor([5.0112e-01, 3.4298e-01, 9.4648e-01, 7.5230e-01, 2.0493e-01, 3.0428e-02,
         9.9269e-01, 9.3894e-01, 5.4725e-01, 1.6793e-01, 4.3259e-01, 8.1498e-02,
         2.1966e-01, 2.0581e-01, 1.5695e-01, 4.2902e-02, 4.7993e-01, 8.5083e-01,
         7.7926e-01, 3.6849e-01, 7.3497e-02, 3.1635e-01, 4.9965e-01, 1.2348e-01,
         5.4839e-02, 8.4317e-01, 3.9206e-01, 1.7243e-01,

#### Instantiating a masked version of our MLP, using the random masks generated:

In [9]:
dict_mask_params_probs

{'fc1': tensor([0.8919, 0.9331, 0.6246, 0.9536, 0.9385, 0.5719, 0.5877, 0.5235, 0.7362,
         0.6367, 0.5371, 0.8091, 0.6430, 0.6810, 0.7832, 0.5742, 0.5045, 0.6753,
         0.9494, 0.6054, 0.8593, 0.6320, 0.7636, 0.8954, 0.6625, 0.9302, 0.9570,
         0.6790, 0.7952, 0.6679, 0.6066, 0.6719, 0.8722, 0.5808, 0.9054, 0.8321,
         0.6886, 0.5780, 0.6424, 0.5479, 0.6319, 0.6395, 0.5034, 0.8391, 0.5948,
         0.9612, 0.5109, 0.9718, 0.8679, 0.6950, 0.8529, 0.6165, 0.7791, 0.9355,
         0.7241, 0.8247, 0.5309, 0.8344, 0.8830, 0.6450, 0.7333, 0.7359, 0.6545,
         0.5027], requires_grad=True),
 'fc2': tensor([5.0112e-01, 3.4298e-01, 9.4648e-01, 7.5230e-01, 2.0493e-01, 3.0428e-02,
         9.9269e-01, 9.3894e-01, 5.4725e-01, 1.6793e-01, 4.3259e-01, 8.1498e-02,
         2.1966e-01, 2.0581e-01, 1.5695e-01, 4.2902e-02, 4.7993e-01, 8.5083e-01,
         7.7926e-01, 3.6849e-01, 7.3497e-02, 3.1635e-01, 4.9965e-01, 1.2348e-01,
         5.4839e-02, 8.4317e-01, 3.9206e-01, 1.7243e-01,

In [10]:
masked_model = MaskedMLP(model, dict_mask_params_probs, probs=True)

masked_model

MaskedMLP(
  (fc1): Linear(in_features=196, out_features=64, bias=True)
  (mask1): MaskLayer()
  (fc2): Linear(in_features=64, out_features=128, bias=True)
  (mask2): MaskLayer()
  (fc3): Linear(in_features=128, out_features=256, bias=True)
  (mask3): MaskLayer()
  (fc4): Linear(in_features=256, out_features=10, bias=True)
)

# Test Function 

In [11]:
from energy_estimator import estimate_model_energy

In [12]:
results = estimate_model_energy(
    model=model,
    board="JetsonNano",
    masks=dict_mask_params_probs,
)

c:\Users\AissaABDELAZIZ\Desktop\package python\energy_estimator\model_energy_aggregation.py:56: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  "energy": float(energy),


In [13]:
for name, info in results["layers"].items():
    print(f"  {name:6s} | {info['layer_key']:12s} | in={info['in']:.0f} out={info['out']:.0f} | energy={info['energy']:.6f} J  ({info['energy_mj']:.4f} mJ)")


  fc1    | linear       | in=196 out=46 | energy=0.000075 J  (0.0745 mJ)
  fc2    | linear       | in=46 out=50 | energy=0.000058 J  (0.0577 mJ)
  fc3    | linear       | in=50 out=119 | energy=0.000080 J  (0.0804 mJ)
  fc4    | linear       | in=119 out=7 | energy=0.000012 J  (0.0115 mJ)


In [14]:
results["total_energy"]=results["total_energy"]* 1000000

In [15]:
results["total_energy"].backward()

In [20]:
print(dict_mask_params_probs['fc4'].grad)

tensor([1.5682, 1.5682, 1.5682, 1.5682, 1.5682, 1.5682, 1.5682, 1.5682, 1.5682,
        1.5682])


In [17]:
import energy_estimator 

print("=" * 60)
print("TEST 1 — available boards and layers")
print("=" * 60)

boards = energy_estimator.list_boards()
print(f"boards : {boards}")

for board in boards:
    keys = energy_estimator.list_layer_keys(board)
    print(f"  {board} → {keys}")

TEST 1 — available boards and layers
boards : ['CoralDevBoard', 'JetsonNano']
  CoralDevBoard → ['linear']
  JetsonNano → ['conv_k3_p0', 'conv_k3_p1', 'conv_k5_p0', 'conv_k5_p1', 'linear']


In [18]:
print("\n" + "=" * 60)
print("TEST 3 — mixed Conv2d + Linear model on JetsonNano")
print("=" * 60)

class MixedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,  32, kernel_size=3, padding=0)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=5, padding=0)
        self.fc1   = nn.Linear(64, 128)
        self.fc2   = nn.Linear(128, 10)

model_mixed = MixedModel()

results = energy_estimator.estimate_model_energy(model_mixed, board="JetsonNano")

for name, info in results["layers"].items():
    print(f"  {name:6s} | {info['layer_key']:12s} | in={info['in']:.0f} out={info['out']:.0f} | energy={info['energy']:.6f} J  ({info['energy_mj']:.4f} mJ)")

print(f"\n  TOTAL energy : {float(results['total_energy']):.6f} J")


TEST 3 — mixed Conv2d + Linear model on JetsonNano
  conv1  | conv_k3_p0   | in=3 out=32 | energy=0.000253 J  (0.2533 mJ)
  conv2  | conv_k5_p0   | in=32 out=64 | energy=0.000670 J  (0.6697 mJ)
  fc1    | linear       | in=64 out=128 | energy=0.000103 J  (0.1033 mJ)
  fc2    | linear       | in=128 out=10 | energy=0.000016 J  (0.0156 mJ)

  TOTAL energy : 0.001042 J
